# 03 — Comprehensive IRS 990-series XML ingestion (separate datasets)

**Purpose:** Inventory and parse **every XML member** in the locally downloaded IRS publication-year 2023–2026 ZIP archives, including Form 990-EZ, 990-PF, and any other return types actually present. Preserve full nested XML content in compressed Parquet without merging it into the existing full Form 990 fraud-label dataset.

**Scope:** Publication years **2023–2026**; **no tax-year filter**. Unlike Notebook 01, this notebook does not discard other form types. It also re-parses full Form 990 as a separate *comprehensive XML* output, while leaving the existing labeled datasets unchanged.

**Outputs:** One Parquet file **per archive × form type**, plus inventory/validation reports. Each filing stores a complete, ordered, namespace-qualified parsed XML tree as JSON (`xml_tree_json`) along with metadata. This retains repeated elements, attributes, text, and nested structure for subsequent form-specific normalization. Original ZIPs remain the authoritative raw source. Parsing every field into one flat table would lose repeated relationships and create an unmanageable schema.

**Caution:** This is an expensive operation (hundreds of thousands of XML files, substantial CPU, disk, and time). Start with the test mode, inspect results, then run full mode. This notebook never modifies Notebook 01/02 outputs.

## 1 — Environment and paths

Run JupyterLab from the project repository and select your `irs990` kernel. Requires `pandas`, `pyarrow`, and standard-library packages. `7zz` is needed only for ZIP compression unsupported by Python (such as Deflate64).

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from xml.etree import ElementTree as ET
import json, os, re, shutil, subprocess, tempfile, time, zipfile
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / ".gitignore").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within irs-990-fraud-detection")
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "staging" / "other_990_xml_partitions"
ANALYTICS = ROOT / "data" / "analytics"
REPORTS = ROOT / "reports" / "other_990_ingestion"
for folder in (OUT, ANALYTICS, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Raw ZIP folder:", RAW)
print("Parquet output:", OUT)

## 2 — Configuration

`RUN_MODE="test"` reads a small number of XML files **from each archive** and writes outputs to a separate `test/` subdirectory. After reviewing test results, change to `RUN_MODE="full"`. Completed full archives are skipped only when their validation report and `_SUCCESS` marker exist. No source ZIP is deleted.

For 7-Zip, set environment variable `SEVEN_ZIP` to its executable path, or install `7zz` on PATH. The fallback path below matches the Mac installation used during Notebook 01; change it if necessary.

In [ ]:
PUBLICATION_YEARS = {2023, 2024, 2025, 2026}
RUN_MODE = "full"               # "test" or "full"
TEST_XML_PER_ARCHIVE = 100      # used only in test mode
BATCH_SIZE = 500               # rows per Parquet write
EXPECTED_ARCHIVE_COUNT = 50    # baseline from Notebook 01; compare with current IRS listing separately

SEVEN_ZIP = os.environ.get("SEVEN_ZIP") or shutil.which("7zz")
if not SEVEN_ZIP:
    mac_candidate = Path.home() / "Library/Mobile Documents/com~apple~CloudDocs/Downloads/7z2605-mac/7zz"
    if mac_candidate.is_file():
        SEVEN_ZIP = str(mac_candidate)

assert RUN_MODE in {"test", "full"}
assert BATCH_SIZE > 0 and TEST_XML_PER_ARCHIVE > 0
MODE_OUT = OUT / ("test" if RUN_MODE == "test" else "full")
MODE_REPORTS = REPORTS / ("test" if RUN_MODE == "test" else "full")
MODE_OUT.mkdir(parents=True, exist_ok=True)
MODE_REPORTS.mkdir(parents=True, exist_ok=True)
print("Mode:", RUN_MODE, "| 7-Zip:", SEVEN_ZIP or "not found")

## 3 — Inventory local archives and check coverage

The ZIP inventory is built from the actual local files, including nested `data/raw/<publication_year>/` directories. It checks ZIP readability and member counts, **not** whether the IRS has published new archives since Notebook 01. Re-run Notebook 01's IRS archive discovery or compare with the live IRS download page before claiming completeness.

In [ ]:
zip_paths = sorted(p for p in RAW.rglob("*.zip") if p.is_file())
archive_rows = []
for path in zip_paths:
    m = re.search(r"(20\d{2})_TEOS_XML", path.name, re.I)
    # Prefer directory year; fall back to filename prefix.
    parent_year = int(path.parent.name) if path.parent.name.isdigit() else None
    publication_year = parent_year or (int(m.group(1)) if m else None)
    if publication_year not in PUBLICATION_YEARS:
        continue
    try:
        with zipfile.ZipFile(path) as z:
            members = [i for i in z.infolist() if i.filename.lower().endswith(".xml") and not i.is_dir()]
            archive_rows.append({"archive_name": path.name, "publication_year": publication_year,
                                 "zip_path": str(path), "zip_bytes": path.stat().st_size,
                                 "xml_members": len(members),
                                 "unsupported_compression": any(i.compress_type not in {0, 8, 12, 14} for i in members),
                                 "zip_error": None})
    except Exception as exc:
        archive_rows.append({"archive_name": path.name, "publication_year": publication_year,
                             "zip_path": str(path), "zip_bytes": path.stat().st_size,
                             "xml_members": None, "unsupported_compression": None,
                             "zip_error": str(exc)})
archives = pd.DataFrame(archive_rows)
if archives.empty:
    raise FileNotFoundError(f"No IRS XML ZIP archives found in {RAW}")
if archives.archive_name.duplicated().any():
    raise ValueError("Duplicate archive filenames detected; resolve before processing")
print("Local archives:", len(archives), "| expected from prior run:", EXPECTED_ARCHIVE_COUNT)
print("Total XML members (ZIP headers):", int(archives.xml_members.fillna(0).sum()))
if len(archives) != EXPECTED_ARCHIVE_COUNT:
    print("WARNING: Local archive count differs from the prior 50-archive baseline")
if archives.zip_error.notna().any():
    raise RuntimeError("Unreadable ZIP(s) detected; inspect inventory")
display(archives.groupby("publication_year").agg(archives=("archive_name", "count"), xml_members=("xml_members", "sum")))
archives.to_csv(ANALYTICS / "03_local_archive_inventory.csv", index=False)

## 4 — Lossless-structure XML serialization and form classification

We retain each element's fully qualified tag, attributes, text, tail, and ordered children. This is a **structural representation** of the parsed XML, not a byte-identical copy (XML comments, declaration formatting, and whitespace serialization may differ). The raw ZIP remains available for byte-level verification.

We classify the return by the top-level child of `<Return>` (e.g., `IRS990`, `IRS990EZ`, `IRS990PF`). Unknown roots are retained as `other` rather than silently excluded. No assumption is made that 990-N or 990-T are present.

In [ ]:
def local_name(tag):
    return str(tag).rsplit("}", 1)[-1].split(":")[-1]


def xml_to_tree(el):
    return {"tag": str(el.tag), "attributes": dict(el.attrib),
            "text": el.text, "tail": el.tail,
            "children": [xml_to_tree(child) for child in el]}


def first_text(root, names):
    for el in root.iter():
        if local_name(el.tag) in names and el.text and el.text.strip():
            return el.text.strip()
    return None


def identify_form(root):
    for child in root:
        tag = local_name(child.tag)
        if tag.startswith("IRS990"):
            return tag, child
    # Keep other return types rather than discarding them.
    for child in root:
        tag = local_name(child.tag)
        if tag not in {"ReturnHeader", "ReturnData"} and tag.startswith("IRS"):
            return tag, child
    for el in root.iter():
        tag = local_name(el.tag)
        if tag.startswith("IRS990"):
            return tag, el
    return "unknown", root


def safe_slug(form_type):
    return re.sub(r"[^a-z0-9_-]+", "_", form_type.lower())[:70] or "unknown"


def filing_row(root, member_name, archive_name, publication_year):
    form_type, _ = identify_form(root)
    header = next((el for el in root.iter() if local_name(el.tag) == "ReturnHeader"), root)
    ein = first_text(header, {"EIN"})
    tax_year_raw = first_text(header, {"TaxYr", "TaxYear"})
    tax_year = int(tax_year_raw) if tax_year_raw and tax_year_raw.isdigit() else None
    return {
        "publication_year": publication_year,
        "archive_name": archive_name,
        "source_xml": member_name,
        "filing_id": Path(member_name).stem.replace("_public", ""),
        "form_type": form_type,
        "ein": ein.zfill(9) if ein and ein.isdigit() else ein,
        "tax_year": tax_year,
        "tax_period_begin": first_text(header, {"TaxPeriodBeginDt", "TaxPeriodBeginDate"}),
        "tax_period_end": first_text(header, {"TaxPeriodEndDt", "TaxPeriodEndDate"}),
        "return_type": first_text(header, {"ReturnTypeCd", "ReturnType"}),
        "organization_name": first_text(header, {"BusinessNameLine1Txt", "BusinessNameLine1", "BusinessName"}),
        "xml_tree_json": json.dumps(xml_to_tree(root), ensure_ascii=False, separators=(",", ":")),
    }

SCHEMA = pa.schema([
    ("publication_year", pa.int16()), ("archive_name", pa.string()),
    ("source_xml", pa.string()), ("filing_id", pa.string()),
    ("form_type", pa.string()), ("ein", pa.string()), ("tax_year", pa.int16()),
    ("tax_period_begin", pa.string()), ("tax_period_end", pa.string()),
    ("return_type", pa.string()), ("organization_name", pa.string()),
    ("xml_tree_json", pa.large_string()),
])

print("Sample form classifier and serializer ready")

## 5 — Streaming archive processing, resumable per archive

Processes one ZIP at a time and writes small batches to Parquet. Files are split by form type **and archive**. Deflate64 archives are extracted to a temporary directory and removed automatically; ensure sufficient free disk space (a previously encountered archive expanded to tens of GB). Writes go to `.part` files and are promoted only after successful processing. If a run fails, rerunning that archive starts it over; previously completed archives are skipped.

**Privacy / correctness:** All fields in public XML, including potentially sensitive contact details, are retained locally. `data/` and `reports/` must remain excluded from Git. Do not publish the raw parsed trees without reviewing their contents.

In [ ]:
def process_archive(archive):
    archive_name = archive["archive_name"]
    publication_year = int(archive["publication_year"])
    zip_path = Path(archive["zip_path"])
    stem = Path(archive_name).stem
    destination = MODE_OUT / stem
    report_path = MODE_REPORTS / f"{stem}.json"
    success_marker = destination / "_SUCCESS"
    if report_path.exists() and success_marker.exists():
        print("SKIP completed:", archive_name)
        return json.loads(report_path.read_text())

    # Remove an incomplete prior attempt, never a completed output.
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    counts = Counter()
    forms = Counter()
    years = Counter()
    errors = []
    writers = {}
    buffers = defaultdict(list)
    start = time.time()

    def flush(form_slug):
        batch = buffers[form_slug]
        if not batch:
            return
        if form_slug not in writers:
            writers[form_slug] = pq.ParquetWriter(
                destination / f"{form_slug}.parquet", SCHEMA, compression="zstd")
        table = pa.Table.from_pylist(batch, schema=SCHEMA)
        writers[form_slug].write_table(table)
        buffers[form_slug] = []

    try:
        with zipfile.ZipFile(zip_path) as z:
            members = [m for m in z.infolist() if m.filename.lower().endswith(".xml") and not m.is_dir()]
            counts["xml_available"] = len(members)
            if RUN_MODE == "test":
                members = members[:TEST_XML_PER_ARCHIVE]
            counts["xml_selected"] = len(members)
            needs_7zip = any(m.compress_type not in {0, 8, 12, 14} for m in members)
            with tempfile.TemporaryDirectory(prefix="irs990_allforms_") as tmp:
                if needs_7zip:
                    if not SEVEN_ZIP or not Path(SEVEN_ZIP).is_file():
                        raise FileNotFoundError("7zz required: set SEVEN_ZIP or install 7zz on PATH")
                    print("7-Zip extraction:", archive_name)
                    proc = subprocess.run([str(SEVEN_ZIP), "x", str(zip_path), f"-o{tmp}", "-y"],
                                          capture_output=True, text=True)
                    if proc.returncode:
                        raise RuntimeError(f"7-Zip failed: {proc.stderr[-1000:]} {proc.stdout[-1000:]}")
                for member in members:
                    counts["xml_attempted"] += 1
                    try:
                        if needs_7zip:
                            # Reject archive member paths escaping the temporary directory.
                            extracted = (Path(tmp) / member.filename).resolve()
                            if not extracted.is_relative_to(Path(tmp).resolve()):
                                raise ValueError("Unsafe archive member path")
                            root = ET.parse(extracted).getroot()
                        else:
                            with z.open(member) as fh:
                                root = ET.parse(fh).getroot()
                        row = filing_row(root, member.filename, archive_name, publication_year)
                        slug = safe_slug(row["form_type"])
                        buffers[slug].append(row)
                        forms[row["form_type"]] += 1
                        years[str(row["tax_year"])] += 1
                        counts["parsed"] += 1
                        if len(buffers[slug]) >= BATCH_SIZE:
                            flush(slug)
                    except Exception as exc:
                        counts["parse_errors"] += 1
                        if len(errors) < 50:
                            errors.append({"source_xml": member.filename, "error": repr(exc)[:400]})
                    if counts["xml_attempted"] % 5000 == 0:
                        print(f"  {archive_name}: {counts['xml_attempted']:,}/{counts['xml_selected']:,}")
        for slug in list(buffers):
            flush(slug)
    except BaseException:
        for writer in writers.values():
            writer.close()
        raise
    else:
        for writer in writers.values():
            writer.close()

    report = {"archive_name": archive_name, "publication_year": publication_year,
              "run_mode": RUN_MODE, "counts": dict(counts),
              "forms": dict(forms), "tax_years": dict(years),
              "errors_first_50": errors, "seconds": round(time.time()-start, 1),
              "output_folder": str(destination.relative_to(ROOT))}
    assert counts["parsed"] + counts["parse_errors"] == counts["xml_selected"]
    report_path.write_text(json.dumps(report, indent=2))
    success_marker.write_text("complete\n")
    print("SAVED:", archive_name, "| parsed:", counts["parsed"], "| errors:", counts["parse_errors"])
    return report

print("Archive processor ready")

## 6 — Run extraction (test first)

Test mode processes up to 100 XML members **per archive**, not necessarily every form type. A successful test only verifies the pipeline; full mode is needed for a complete form inventory. Processing all 50 archives may take hours. Keep the Mac awake if necessary (e.g. run `caffeinate -dims` in a separate terminal).

In [ ]:
results = []
for row in archives.sort_values(["publication_year", "archive_name"]).to_dict("records"):
    results.append(process_archive(row))
print("Completed or resumed:", len(results), "archives")

## 7 — Validate counts, types, and tax-year coverage

The full run is complete only when every local ZIP has a successful report, every XML member was attempted, and all parse errors are understood. Nonzero errors are surfaced rather than silently discarded. This audit does **not** prove that the local ZIP list matches newly released IRS archives; that is a separate live-source check.

In [ ]:
report_paths = sorted(MODE_REPORTS.glob("*.json"))
reports = [json.loads(p.read_text()) for p in report_paths]
summary = pd.DataFrame([{
    "archive_name": r["archive_name"], "publication_year": r["publication_year"],
    "xml_available": r["counts"].get("xml_available", 0),
    "xml_selected": r["counts"].get("xml_selected", 0),
    "parsed": r["counts"].get("parsed", 0),
    "parse_errors": r["counts"].get("parse_errors", 0),
    "form_types": ", ".join(sorted(r["forms"])),
} for r in reports])
form_counts = Counter()
tax_counts = Counter()
for r in reports:
    form_counts.update(r["forms"])
    tax_counts.update(r["tax_years"])
print("Archives reported:", len(reports), "/", len(archives))
print("XML parsed:", sum(form_counts.values()))
print("Parse errors:", int(summary.parse_errors.sum()) if not summary.empty else 0)
display(pd.DataFrame(sorted(form_counts.items()), columns=["form_type", "filings"]))
display(pd.DataFrame(sorted(tax_counts.items()), columns=["tax_year", "filings"]))
display(summary)
summary.to_csv(ANALYTICS / f"03_other_990_archive_summary_{RUN_MODE}.csv", index=False)
pd.DataFrame(sorted(form_counts.items()), columns=["form_type", "filings"]).to_csv(
    ANALYTICS / f"03_other_990_form_inventory_{RUN_MODE}.csv", index=False)

if RUN_MODE == "full":
    assert len(reports) == len(archives), "Some archives have no report"
    assert (summary.xml_available == summary.xml_selected).all(), "Some XML members were not selected"
    assert (summary.parsed + summary.parse_errors == summary.xml_selected).all(), "Count reconciliation failed"
    if summary.parse_errors.sum():
        print("WARNING: Some XML files failed parsing; inspect per-archive JSON reports")
    else:
        print("All locally inventoried XML members parsed successfully")
else:
    print("TEST MODE: results are partial; change RUN_MODE to 'full' for comprehensive ingestion")

## 8 — Read an extracted record and recover nested fields

This demonstrates how to inspect the retained full tree without assuming every form shares the same schema. The example reads a single Parquet file. For downstream analysis, create separate **normalized tables** for repeating grants, compensation, addresses, etc., keyed by `filing_id` plus a stable source identifier.

In [ ]:
parquets = sorted(MODE_OUT.rglob("*.parquet"))
if not parquets:
    print("No extracted Parquet files yet")
else:
    example = pq.ParquetFile(parquets[0])
    first_batch = next(example.iter_batches(batch_size=1))
    record = first_batch.to_pylist()[0]
    tree = json.loads(record["xml_tree_json"])
    print("File:", parquets[0].relative_to(ROOT))
    print("Form:", record["form_type"], "| tax year:", record["tax_year"])
    print("Root:", tree["tag"], "| root children:", len(tree["children"]))
    print("All nested content preserved in xml_tree_json")

## 9 — Next steps and guardrails

- **No automatic merging:** These comprehensive datasets remain in `data/staging/other_990_xml_partitions/`, separate from the existing labeled Form 990 Parquet and organization-year candidates.
- **No false negatives:** Returns without the full Form 990 diversion question must **not** receive `fraud_label=0`.
- **Source provenance:** Keep `archive_name`, `source_xml`, `filing_id`, publication year, form type, and tax year.
- **Completeness:** Compare local archive inventory against the **current** IRS publication page periodically; 2026 may gain additional releases. Notebook 01's 50-archive count is only the baseline from the prior run.
- **Data volume:** The `xml_tree_json` columns can be very large. Avoid loading all of them into pandas simultaneously; use Parquet column projection and per-archive iteration.
- **Future Notebook 04:** Extract targeted, schema-aware financial and governance tables from these trees and/or raw XML, and join to the original Form 990 labels by verified filing identity. Preserve other form types separately.
- **Git:** Confirm `data/` and `reports/` are ignored; do not commit these large outputs.

**Note:** This notebook ingests *all fields of the forms found in the downloaded ZIPs*, not every IRS 990-series product published through separate distribution channels. If other products are required, acquire and inventory their distinct source datasets explicitly.